In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Environment & Path Configuration
# Auto-detects Google Colab vs Local Jupyter and sets all path variables.
# Run this cell FIRST before any other cell.
# ─────────────────────────────────────────────────────────────────────────────
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    import subprocess
    repo_path = Path('/content/amazon-ml-challenge-2026')
    if not repo_path.exists():
        subprocess.run(
            ['git', 'clone',
             'https://github.com/SmithC05/amazon-ml-challenge-2026.git',
             str(repo_path)], check=True)
    os.chdir(repo_path)
    sys.path.insert(0, str(repo_path))
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT   = Path('/content/drive/MyDrive/Amazon ML Challenge 2026')
    REPO_ROOT    = repo_path
    DATASET_ROOT = DRIVE_ROOT / '01_Dataset'
    TRAIN_DIR    = DATASET_ROOT / ' raw' / 'train'
    CACHE_DIR    = DATASET_ROOT / 'processed' / 'm2_cache'
    GT_PATH      = TRAIN_DIR / 'train_ground_truth.tsv'
    OUTPUT_DIR   = Path('/content')
else:
    _nb_dir = Path(globals().get('__vsc_ipynb_file__',
                   globals().get('__file__', ''))).resolve().parent
    REPO_ROOT = _nb_dir.parent if _nb_dir.name == 'notebooks' else _nb_dir
    if not (REPO_ROOT / 'src').exists():
        REPO_ROOT = Path.cwd()
    sys.path.insert(0, str(REPO_ROOT / 'src'))
    sys.path.insert(0, str(REPO_ROOT))
    os.chdir(REPO_ROOT)
    DATASET_ROOT = REPO_ROOT / 'dataset'
    TRAIN_DIR    = DATASET_ROOT / 'raw' / 'train'
    CACHE_DIR    = DATASET_ROOT / 'processed' / 'm2_cache'
    GT_PATH      = TRAIN_DIR / 'train_ground_truth.tsv'
    OUTPUT_DIR   = REPO_ROOT / 'output'

print(f"Environment : {'Google Colab' if IS_COLAB else 'Local Jupyter'}")
print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"DATASET_ROOT: {DATASET_ROOT}")
print(f"TRAIN_DIR   : {TRAIN_DIR}")
print(f"CACHE_DIR   : {CACHE_DIR}")
print(f"GT_PATH     : {GT_PATH}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"cache exists : {CACHE_DIR.exists()}")
print(f"gt exists    : {GT_PATH.exists()}")


# Notebook 06 — Candidate Generation (Blocking)
## Member 4 Deliverable

---

### Role boundary

| Owner | Responsibility |
|---|---|
| **Member 2** | `src/preprocess.py`, `src/cache.py` — normalization + Parquet cache |
| **Member 4** | `src/candidate_generation.py` — blocking / candidate generation |
| **Member 3** | `src/train.py`, `src/predict.py` — baseline model |
| **Team lead** | Advanced model experiments, final submission |

This notebook:
1. Loads M2-normalized data from Parquet cache
2. Benchmarks each blocking strategy individually
3. Combines all blocks and evaluates recall / volume / reduction
4. Generates `output/candidate_pairs.tsv` in the official format
5. Validates the output and prints final statistics

---

## 1. Setup

In [ ]:
import sys, time, json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# REPO_ROOT, sys.path set by env_path_config cell

# M2 cache
from cache import build_cache, load_all_cache, cache_exists

# M4 blocking
from candidate_generation import (
    generate_name_exact_candidates,
    generate_name_candidates,
    generate_address_candidates,
    generate_prefix_candidates,
    generate_country_token_candidates,
    generate_fuzzy_candidates,
    generate_candidates,
    evaluate_candidates,
)

# Format contract
from candidates import to_official_format, write_candidates, validate_format

# M3 train helper for truth_map
from train import build_truth_map

# Paths — adjust DATA_DIR to your Drive/local path
DATA_DIR  = TRAIN_DIR  # set by env_path_config cell above
CACHE_DIR = REPO_ROOT / 'cache'
OUTPUT_DIR = REPO_ROOT / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

print('Setup OK')

## 2. Build / Load M2 Cache

In [ ]:
# Build cache if not present (only runs once per machine)
if not cache_exists(CACHE_DIR, 'train'):
    print('Building M2 cache...')
    build_cache(DATA_DIR, CACHE_DIR, split='train', force=False)

t0 = time.perf_counter()
s1, s2, s3 = load_all_cache(CACHE_DIR, split='train')
load_time = time.perf_counter() - t0

print(f'Loaded from cache in {load_time:.2f}s')
print(f'S1={len(s1):,}  S2={len(s2):,}  S3={len(s3):,}')

# Also load ground truth for evaluation
gt = pd.read_csv(DATA_DIR / 'train_ground_truth.tsv', sep='\t')
truth_map = build_truth_map(gt)
print(f'GT labeled S1 entities: {len(truth_map):,}')

## 3. Inspect Normalized Data

In [ ]:
print('S1 sample (normalized):')
display(s1[['entity_id','business_name_norm','business_address_norm','country']].head())

print(f"\nAddress empty: S1={s1['address_is_empty'].sum():,}  S2={s2['address_is_empty'].sum():,}  S3={s3['address_is_empty'].sum():,}")
print(f"Name tokens (S1 median): {s1['name_tokens'].median():.0f}")

## 4. Individual Block Benchmarks

Each block runs independently so we can measure its contribution.

In [ ]:
block_results = {}

def run_block(name, fn, *args):
    t0 = time.perf_counter()
    result = fn(*args)
    elapsed = time.perf_counter() - t0
    n = len(result)
    print(f'  [{name:15s}]  pairs={n:>8,}  time={elapsed:.2f}s')
    return result

print('--- S2 blocking ---')
b1_s2 = run_block('exact/S2',        generate_name_exact_candidates,    s1, s2)
b2_s2 = run_block('token/S2',        generate_name_candidates,          s1, s2)
b3_s2 = run_block('address/S2',      generate_address_candidates,       s1, s2)
b4_s2 = run_block('prefix/S2',       generate_prefix_candidates,        s1, s2)
b5_s2 = run_block('country_tok/S2',  generate_country_token_candidates, s1, s2)
b6_s2 = run_block('fuzzy/S2',        generate_fuzzy_candidates,         s1, s2)

print('\n--- S3 blocking ---')
b1_s3 = run_block('exact/S3',        generate_name_exact_candidates,    s1, s3)
b2_s3 = run_block('token/S3',        generate_name_candidates,          s1, s3)
b3_s3 = run_block('address/S3',      generate_address_candidates,       s1, s3)
b4_s3 = run_block('prefix/S3',       generate_prefix_candidates,        s1, s3)
b5_s3 = run_block('country_tok/S3',  generate_country_token_candidates, s1, s3)
b6_s3 = run_block('fuzzy/S3',        generate_fuzzy_candidates,         s1, s3)

In [ ]:
# Evaluate recall per block (combined S2+S3 for each)
block_defs = [
    ('1-exact',         b1_s2, b1_s3),
    ('2-token',         b2_s2, b2_s3),
    ('3-address',       b3_s2, b3_s3),
    ('4-prefix',        b4_s2, b4_s3),
    ('5-country-token', b5_s2, b5_s3),
    ('6-fuzzy',         b6_s2, b6_s3),
]

summary_rows = []
for bname, bs2, bs3 in block_defs:
    combined = pd.concat([bs2, bs3], ignore_index=True).drop_duplicates(
        subset=['source1_entity_id','candidate_entity_id'])
    off = to_official_format(combined, all_s1_ids=s1['entity_id'].tolist())
    m = evaluate_candidates(off, truth_map, len(s2), len(s3))
    summary_rows.append({
        'block': bname,
        'pairs': m['total_candidate_ids'],
        'recall': m['candidate_recall'],
        'lost': m['true_matches_lost'],
        'avg_per_s1': m['avg_per_s1'],
        'reduction': m['reduction_ratio'],
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

## 5. Combined Blocking — All Strategies Unioned

In [ ]:
print('Running full candidate generation pipeline...')
t0 = time.perf_counter()
internal_df, official_df = generate_candidates(s1, s2, s3, verbose=True)
elapsed = time.perf_counter() - t0
print(f'\nTotal time: {elapsed:.2f}s')

## 6. Evaluation — Recall / Volume / Reduction

In [ ]:
metrics = evaluate_candidates(official_df, truth_map, n_s2=len(s2), n_s3=len(s3))
print('Candidate Generation Metrics:')
print(json.dumps(metrics, indent=2))

In [ ]:
# Candidate count distribution
counts = official_df['candidate_entity_ids'].apply(
    lambda x: len(x.split(',')) if isinstance(x, str) and x.strip() else 0
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(counts[counts > 0], bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Candidates per S1 entity')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of candidate counts (excluding zeros)')
axes[0].grid(alpha=0.3)

axes[1].hist(counts[counts > 0].clip(upper=100), bins=50, color='coral', edgecolor='white')
axes[1].set_xlabel('Candidates per S1 entity (clipped at 100)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution (clipped at 100)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Avg: {counts.mean():.1f}  Median: {counts.median():.0f}  Max: {counts.max()}")
print(f"Zero-candidate S1 entities: {(counts==0).sum():,}")

## 7. Write Official candidate_pairs.tsv

In [ ]:
# Validate before writing
errors = validate_format(official_df)
if errors:
    for e in errors:
        print(f'ERROR: {e}')
else:
    print('Format validation: PASSED')

out_path = OUTPUT_DIR / 'candidate_pairs.tsv'
write_candidates(official_df, str(out_path))

# Verify it round-trips correctly
check = pd.read_csv(out_path, sep='\t')
assert list(check.columns) == ['source1_entity_id', 'candidate_entity_ids'], 'Column mismatch'
print(f'Verified: {len(check):,} rows, columns={list(check.columns)}')

## 8. Additional Validation Checks

In [ ]:
# Check: no S1 IDs in candidate list
all_cands = ','.join(official_df['candidate_entity_ids'].dropna().tolist())
s1_in_cands = [x for x in all_cands.split(',') if x.startswith('S1-')]
print(f'S1 IDs in candidate list: {len(s1_in_cands)} (must be 0)')
assert not s1_in_cands, 'S1 IDs found in candidate list!'

# Check: all S1 entities present exactly once
assert official_df['source1_entity_id'].nunique() == len(official_df), 'Duplicate S1 rows!'
s1_ids_set = set(s1['entity_id'])
output_ids_set = set(official_df['source1_entity_id'])
missing = s1_ids_set - output_ids_set
extra   = output_ids_set - s1_ids_set
print(f'Missing S1 IDs: {len(missing)} (must be 0)')
print(f'Extra S1 IDs  : {len(extra)} (must be 0)')
assert not missing and not extra

# Check: no intra-row duplicate candidate IDs
intra_dupes = 0
for _, row in official_df.iterrows():
    raw = row['candidate_entity_ids']
    if isinstance(raw, str) and raw.strip():
        ids = raw.split(',')
        if len(ids) != len(set(ids)):
            intra_dupes += 1
print(f'Rows with intra-row duplicate IDs: {intra_dupes} (must be 0)')
assert intra_dupes == 0

print('\nAll validation checks PASSED.')

In [ ]:
# Verify M3 can parse the file without format changes
from train import build_pair_rows, build_truth_map

cands_check = pd.read_csv(out_path, sep='\t')
print(f'candidate_pairs.tsv columns: {list(cands_check.columns)}')

# M3 expects: source1_entity_id, candidate_entity_ids
assert 'source1_entity_id'   in cands_check.columns
assert 'candidate_entity_ids' in cands_check.columns
print('M3 compatibility check: PASSED')

---
## 9. Final Statistics Summary


In [ ]:
print('=' * 55)
print('CANDIDATE GENERATION — FINAL STATISTICS')
print('=' * 55)
print(f"Blocking strategies : 6 (exact, token, address, prefix, country-token, fuzzy)")
print(f"Total S1 entities   : {metrics['n_s1']:,}")
print(f"Total candidate IDs : {metrics['total_candidate_ids']:,}")
print(f"Avg candidates/S1   : {metrics['avg_per_s1']}")
print(f"Median candidates   : {metrics['median_per_s1']}")
print(f"Max candidates/S1   : {metrics['max_per_s1']:,}")
print(f"S1 with 0 candidates: {metrics['n_zero_candidates']:,}")
print(f"Cross-product size  : {metrics['cross_product_size']:,}")
print(f"Reduction ratio     : {metrics['reduction_ratio']:.4f}")
print(f"True matches found  : {metrics['true_matches_found']:,}")
print(f"True matches LOST   : {metrics['true_matches_lost']:,}")
print(f"Candidate recall    : {metrics['candidate_recall']:.4f}")
print('=' * 55)
print(f"Output: output/candidate_pairs.tsv")
print(f"Format: source1_entity_id | candidate_entity_ids (comma-sep)")
print('This file is the FINAL candidate set fed to the M3 matching model.')